# Temporal Detection Evaluation

Evaluate cat and person tracks using temporal IoU, precision, recall and F1.

## Configure notebook

In [ ]:
from nb_utils import setup_notebook

PROJECT_ROOT = setup_notebook()

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from matplotlib import pyplot as plt
from optuna.trial import TrialState

import batch_run
from config import SETTINGS_PATH_OVERRIDE, settings
from nb_utils import (
    PERSON_LABEL_ID,
    VideoInfo,
    annotation_intervals,
    is_video_annotated,
    load_annotations,
)
from utils import get_video_paths

In [ ]:
# define constants
N_TRIALS = 100

In [ ]:
# define paths
METADATA_DIR = PROJECT_ROOT / "object_clips" / "metadata"
LABELS_PATH = PROJECT_ROOT / "datasets" / "behaviour_dataset" / "behaviour_labels.json"
OPTIMISATION_CACHE_DIR = PROJECT_ROOT / ".cache" / "temporal_detection"

## Load target videos

In [ ]:
# all processed or annotated videos should be a subset of target videos
video_paths = get_video_paths(mock_inputs_long=True)
video_paths.sort(key=lambda p: p.stat().st_size)
video_info = {p.name: VideoInfo(p) for p in video_paths}

## Define functions

In [ ]:
def union_person_intervals(intervals: list[dict]) -> list[dict]:
    """union person presence without changing individual cat intervals"""
    result = [r.copy() for r in intervals if r["class"] == "cat"]
    people = defaultdict(list)
    for interval in intervals:
        if interval["class"] == "person":
            people[interval["video"]].append(interval)
    for windows in people.values():
        merged = []
        for interval in sorted(windows, key=lambda r: r["start"]):
            if merged and interval["start"] <= merged[-1]["end"] + 1:
                merged[-1]["end"] = max(merged[-1]["end"], interval["end"])
            else:
                merged.append({**interval, "cat_id": "person"})
        result.extend(merged)
    return result


def evaluate_temporal_predictions(
    ground_truth: list[dict],
    predictions: list[dict],
    video_info: dict[str, VideoInfo],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """calculate temporal metrics by class"""
    rows = []
    grouped = defaultdict(list)

    # prepare intervals for identity-specific scoring
    for role, intervals in (("truth", ground_truth), ("prediction", predictions)):
        for int in union_person_intervals(intervals):
            dur_seconds = (int["end"] - int["start"] + 1) / video_info[int["video"]].fps
            row = int | {
                "role": role,
                "duration_seconds": dur_seconds,
                "credited_seconds": 0,
            }
            rows.append(row)
            grouped[(row["video"], row["class"], row.get("cat_id"))].append(row)

    # credit uniquely matched intervals
    for group_key, group_int in grouped.items():
        video_name = group_key[0]
        bounds = sorted({n for r in group_int for n in (r["start"], r["end"] + 1)})
        for seg_start, seg_end in zip(bounds[:-1], bounds[1:]):
            active = [r for r in group_int if r["start"] <= seg_start <= r["end"]]
            if sum(r["role"] == "prediction" for r in active) == 1 and any(
                r["role"] == "truth" for r in active
            ):
                seconds = (seg_end - seg_start) / video_info[video_name].fps
                for interval in active:
                    interval["credited_seconds"] += seconds

    # calculate metrics by class
    coverage = pd.DataFrame(rows)
    coverage["coverage"] = coverage["credited_seconds"] / coverage["duration_seconds"]
    metrics = []
    for cls in ("cat", "person"):
        mask = coverage["class"] == cls
        gt = coverage.loc[mask & (coverage["role"] == "truth"), "coverage"]
        pred = coverage.loc[mask & (coverage["role"] == "prediction"), "coverage"]
        metrics.append(
            {
                "class": cls,
                "ground-truth intervals": len(gt),
                "predicted intervals": len(pred),
                "precision": pred.mean(),
                "recall": gt.mean(),
            }
        )
    metrics = pd.DataFrame(metrics).set_index("class")
    num = 3 * metrics["precision"] * metrics["recall"]
    denum = 2 * metrics["precision"] + metrics["recall"]
    metrics["F-beta"] = num / denum

    return metrics, coverage


def build_ground_truth(
    annotations: list[dict], ann_video_info: dict[str, VideoInfo]
) -> list[dict]:
    """merge annotation windows into class-specific presence intervals"""
    ground_truth = []
    for video_name, video in ann_video_info.items():
        intervals = annotation_intervals(annotations, video.hashes, video_name)
        for cat_id, start, end, _ in intervals:
            ground_truth.append(
                {
                    "video": video_name,
                    "class": "person" if cat_id == PERSON_LABEL_ID else "cat",
                    "cat_id": cat_id,
                    "start": start,
                    "end": end,
                }
            )
    return union_person_intervals(ground_truth)


def load_tracks(metadata_dir: Path) -> list[dict]:
    """load exported tracks from the metadata directory"""
    tracks = [joblib.load(p) for p in metadata_dir.glob("track-*.joblib")]
    if any(t["object_name"] == "cat" and not t.get("cat_id") for t in tracks):
        raise ValueError("A cat track is missing cat_id")
    return tracks


def load_prediction_intervals(
    tracks: list[dict], video_info: dict[str, VideoInfo]
) -> list[dict]:
    """convert exported tracks into source-frame intervals"""
    # build hash map for each video
    video_hash_indices = {
        n: {h: i for i, h in enumerate(v.hashes)} for n, v in video_info.items()
    }

    # calculate index based intervals
    predictions = []
    for track in tracks:
        name = Path(track["video_path"]).name
        if name in video_info:
            hash_indices = video_hash_indices[name]
            frame_indices = [
                hash_indices[h] for h in track["frame_hashes"] if h in hash_indices
            ]
            if frame_indices:
                predictions.append(
                    {
                        "video": name,
                        "class": track["object_name"],
                        "cat_id": track["cat_id"],
                        "start": min(frame_indices),
                        "end": max(frame_indices),
                    }
                )

    return predictions

## Load ground truth

In [ ]:
# identify labelled target videos and report unlabelled videos
annotations = load_annotations(LABELS_PATH)
ann_video_info = {
    n: v for n, v in video_info.items() if is_video_annotated(v, annotations)
}
unlabelled_videos = set(video_info) - set(ann_video_info)
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")
ground_truth = build_ground_truth(annotations, ann_video_info)
print(f"Ground-truth intervals: {len(ground_truth)}")

## Run batch processing

In [ ]:
# apply evaluation-only settings to the child process
try:
    SETTINGS_PATH_OVERRIDE.write_text('LOG_LEVEL = "ERROR"\nEXCLUDED_OBJECTS = ""\n')
    return_code = batch_run.main(video_paths, clear_output=True)
finally:
    SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)

## Load predicted tracks

In [ ]:
# load exported tracks and report unprocessed target videos
tracks = load_tracks(METADATA_DIR)
proc_video_names = {
    n for t in tracks if (n := Path(t["video_path"]).name) in video_info
}
proc_video_info = {n: v for n, v in video_info.items() if n in proc_video_names}
unprocessed_videos = set(video_info) - set(proc_video_info)
print(f"Unprocessed videos: {len(unprocessed_videos)}")
for video_name in sorted(unprocessed_videos):
    print(f"  {video_name}")

# convert exported tracks into source-frame intervals
predictions = load_prediction_intervals(tracks, proc_video_info)

# report the number of predicted intervals
print(f"Predicted intervals: {len(predictions)}")

## Evaluate predictions

In [ ]:
# calculate identity-specific coverage and class scores
metrics, coverage = evaluate_temporal_predictions(
    ground_truth, predictions, ann_video_info
)
agg_col = {
    "ground-truth intervals": "sum",
    "predicted intervals": "sum",
    "precision": "mean",
    "recall": "mean",
    "F-beta": "mean",
}
pd.concat([metrics, metrics.agg(agg_col).to_frame("all").T]).round(3)

## Plot per-video timelines

In [ ]:
# find videos with uncovered truth or uncredited predictions
video_names_with_errors = sorted(
    coverage.loc[~np.isclose(coverage["coverage"], 1), "video"].unique()
)

# compare truth and scored predictions for videos with errors
if video_names_with_errors:
    fig, axes = plt.subplots(
        len(video_names_with_errors),
        1,
        figsize=(14, 2.5 * len(video_names_with_errors)),
        squeeze=False,
    )

    # iterate over videos and categories and intervals
    for v_ind, (axis, video_name) in enumerate(
        zip(axes[:, 0], video_names_with_errors)
    ):
        for lane, (class_name, role, colour) in enumerate(
            [
                ("cat", "truth", "tab:green"),
                ("cat", "prediction", "tab:blue"),
                ("person", "truth", "tab:orange"),
                ("person", "prediction", "tab:red"),
            ]
        ):
            lane_intervals = coverage[
                (coverage["video"] == video_name)
                & (coverage["class"] == class_name)
                & (coverage["role"] == role)
            ].to_dict("records")
            for c_ind, interval in enumerate(lane_intervals):

                # calculate offset for overlapping intervals
                overlapping_before = sum(
                    interval["start"] <= x["end"] and x["start"] <= interval["end"]
                    for x in lane_intervals[:c_ind]
                )
                overlaps_later = any(
                    interval["start"] <= x["end"] and x["start"] <= interval["end"]
                    for x in lane_intervals[c_ind + 1 :]
                )
                offset = 0
                if overlapping_before or overlaps_later:
                    offset = 0.15 * (-1) ** (overlapping_before + 1)

                # plot the intervals
                axis.plot(
                    [interval["start"], interval["end"]],
                    [lane + offset, lane + offset],
                    color=colour,
                    linewidth=3,
                )

        # annotate the axis
        ticks = ["cat GT", "cat tracks", "person GT", "person tracks"]
        axis.set_yticks(range(4), ticks)
        axis.set_title(f"{v_ind}:   {video_name}")
        axis.set_xlabel("Source frame")
        axis.grid(axis="x", alpha=0.25)
    fig.tight_layout()

## Optimisation

In [ ]:
# define the full parameter search and baseline
WEIGHT_PARAMETER_NAMES = (
    "TRACK_MATCH_WEIGHT_IOU",
    "TRACK_MATCH_WEIGHT_CENTROID",
    "TRACK_MATCH_WEIGHT_CONFIDENCE",
    "TRACK_MATCH_WEIGHT_VISUAL",
    "TRACK_MATCH_WEIGHT_RECENCY",
    "TRACK_MATCH_WEIGHT_AGE",
    "TRACK_MATCH_WEIGHT_SIZE",
    "TRACK_MATCH_WEIGHT_IDENTITY",
)
FLOAT_SEARCH_SPACE = {
    "CONF": (0.1, 0.35, 0.05),
    "TRACK_NEW_CONF_THRESHOLD": (0.25, 0.6, 0.05),
    "TRACK_NEW_DUR": (1, 5, 0.5),
    "TRACK_NEW_MIN_DETECTION_RATIO": (0.3, 0.8, 0.1),
    "TRACK_NEW_MIN_CONF_RATIO": (0.1, 0.5, 0.05),
    "TRACK_STALE_DUR": (3, 15, 1),
    "TRACK_MATCH_THRESHOLD": (0.4, 0.7, 0.05),
    "NMS_IOU_THRESHOLD": (0.2, 0.6, 0.1),
    "NMS_IOM_THRESHOLD": (0.6, 0.95, 0.05),
    **{n: (0.5, 5, 0.5) for n in WEIGHT_PARAMETER_NAMES},
}
BASELINE_PARAMS = {n: float(getattr(settings, n)) for n in FLOAT_SEARCH_SPACE}
BASELINE_PARAMS["VERIFY_DETECTIONS"] = bool(settings.VERIFY_DETECTIONS)
PENALTY_STEPS = {n: b[2] for n, b in FLOAT_SEARCH_SPACE.items()}
PENALTY_STEPS["VERIFY_DETECTIONS"] = 1

In [ ]:
# optimisation functions
def suggest_parameters(trial: optuna.trial.Trial) -> dict:
    """suggest a complete tracking configuration"""
    params = {
        n: trial.suggest_float(n, l, h, step=s)
        for n, (l, h, s) in FLOAT_SEARCH_SPACE.items()
    }
    params["VERIFY_DETECTIONS"] = trial.suggest_categorical(
        "VERIFY_DETECTIONS", [False, True]
    )
    return params


def parameter_penalty(params: dict) -> float:
    """calculate the per-parameter cost of deviating from baseline"""
    steps = [
        abs(float(params[n]) - float(BASELINE_PARAMS[n])) / s
        for n, s in PENALTY_STEPS.items()
    ]
    return 0.01 * np.mean(steps)


def run_optimisation() -> tuple[optuna.study.Study, pd.DataFrame]:
    """run or resume the persistent per-video study"""
    # create persistent study storage
    OPTIMISATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    storage_url = f"sqlite:///{(OPTIMISATION_CACHE_DIR / 'study.sqlite3').resolve()}"
    study = optuna.create_study(
        study_name="temporal_detection_f_beta_per_video",
        storage=storage_url,
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    # retry failed and still-running trials
    retry_trials = study.get_trials(states=(TrialState.FAIL, TrialState.RUNNING))
    for t in retry_trials:
        study.enqueue_trial(t.params)

    def objective(trial: optuna.trial.Trial) -> float:
        """evaluate and optimise one tracking configuration"""
        parameters = suggest_parameters(trial)
        predictions = []

        def evaluate_progress(video_count: int) -> float:
            """evaluate the trial on the processed annotated video prefix"""
            video_sub = {
                p.name: ann_video_info[p.name]
                for p in video_paths[:video_count]
                if p.name in ann_video_info
            }
            gt_sub = [i for i in ground_truth if i["video"] in video_sub]
            pred_sub = [i for i in predictions if i["video"] in video_sub]
            trial_metrics, _ = evaluate_temporal_predictions(
                gt_sub, pred_sub, video_sub
            )
            score = trial_metrics["F-beta"].fillna(0).mean()
            trial.report(score, step=video_count)
            return score

        # create a per-trial settings file for the child processes
        override = {"LOG_LEVEL": "ERROR", "EXCLUDED_OBJECTS": "", **parameters}
        override_text = "\n".join(f"{n} = {json.dumps(v)}" for n, v in override.items())
        SETTINGS_PATH_OVERRIDE.write_text(f"{override_text}\n")

        # process and evaluate each video
        try:
            for index, video_path in enumerate(video_paths):
                message = f"Trial {trial.number}, video {index + 1}/{len(video_paths)}: {video_path.name}"
                print(message)
                return_code = batch_run.main([video_path], clear_output=True)
                if return_code == 0:
                    tracks = load_tracks(METADATA_DIR)
                    video_predictions = load_prediction_intervals(
                        tracks, {video_path.name: video_info[video_path.name]}
                    )
                    predictions.extend(video_predictions)
                else:
                    message = f"Skipping {video_path.name}: retries exhausted (code {return_code}); no detections"
                    print(message)

                score = evaluate_progress(index + 1)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            trial.set_user_attr("pre-penalty macro-averaged F-beta", score)
            return score - parameter_penalty(parameters)
        finally:
            SETTINGS_PATH_OVERRIDE.unlink(missing_ok=True)

    # continue to the configured count of complete or pruned trials
    completed_trials = sum(
        t.state in (TrialState.COMPLETE, TrialState.PRUNED) for t in study.trials
    )
    remaining_trials = max(N_TRIALS - completed_trials, 0)
    study.optimize(objective, n_trials=remaining_trials)

    # display the best fully evaluated candidate
    trial_results = study.trials_dataframe(
        attrs=("number", "state", "value", "params", "user_attrs"), multi_index=False
    ).rename(
        columns={
            "number": "trial",
            "value": "objective score",
            "user_attrs_pre-penalty macro-averaged F-beta": "pre-penalty macro-averaged F-beta",
        }
    )
    trial_results = trial_results.sort_values("objective score", ascending=False)
    return study, trial_results

In [ ]:
# run optimisation and display results
study, optimisation_trial_results = run_optimisation()
display(
    optimisation_trial_results.loc[
        optimisation_trial_results["state"] == "COMPLETE"
    ].head()
)
display(pd.Series(study.best_params))